In [ ]:
import random
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import plotly.graph_objects as go

import open_clip
from PIL import Image

from tqdm import tqdm

from qdrant_client import QdrantClient
from qdrant_client.models import QueryRequest

BACKEND_DIR = Path("..").resolve()
DATASET_DIR = BACKEND_DIR / "dataset"
TRAIN_DIR = DATASET_DIR / "train_ready"
VAL_DIR = DATASET_DIR / "val_ready"

QDRANT_URL = "http://localhost:6333"
COLLECTION = "plant_images"

MODEL_NAME = "hf-hub:imageomics/bioclip"
BATCH_SIZE = 64
NUM_WORKERS = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BROJ_SLIKA = 1000
BROJ_SKUPOVA = 29
SEED = 42
K_MAX = 3000

client = QdrantClient(url=QDRANT_URL, timeout=60)

print("DEVICE =", DEVICE)
print("Tačaka u kolekciji:", client.count(COLLECTION).count)

DEVICE = cuda


Tačaka u kolekciji: 135877


In [2]:
MAPPING_CSV = DATASET_DIR / "specie_id - scientific_name mapping.csv"

species_id_name_df: pd.DataFrame = pd.read_csv(MAPPING_CSV, dtype=str)
species_id_name_dict: dict[str, str] = dict(zip(species_id_name_df["specie_id"], species_id_name_df["scientific_name"]))

print("Vrsta u mapiranju:", len(species_id_name_dict))

Vrsta u mapiranju: 324


In [ ]:
val_paths: list[Path] = sorted(VAL_DIR.glob("*/*.jpg"))
assert val_paths, f"Nema .jpg slika u {VAL_DIR}/<specie_id>/ — proveri putanju i strukturu foldera"
print("Slika u val_ready:", len(val_paths))

train_names = {p.name for p in TRAIN_DIR.glob("*/*.jpg")}
leaked = [p for p in val_paths if p.name in train_names]
print("Od toga i u train_ready (izbacuju se):", len(leaked))
val_paths = [p for p in val_paths if p.name not in train_names]

unknown = {p.parent.name for p in val_paths} - set(species_id_name_dict)
assert not unknown, f"Vrste kojih nema u mapiranju: {unknown}"

assert len(val_paths) >= BROJ_SKUPOVA * BROJ_SLIKA, \
    f"Ima {len(val_paths)} slika, a treba {BROJ_SKUPOVA * BROJ_SLIKA} — smanji BROJ_SKUPOVA ili BROJ_SLIKA"

shuffled = val_paths.copy()
random.Random(SEED).shuffle(shuffled)

val_records: list[dict] = []
for skup in range(1, BROJ_SKUPOVA + 1):
    for path in sorted(shuffled[(skup - 1) * BROJ_SLIKA : skup * BROJ_SLIKA]):
        specie_id = path.parent.name
        val_records.append({
            "skup": skup,
            "path": path,
            "image_path": path.relative_to(DATASET_DIR).as_posix(),
            "specie_id": specie_id,
            "scientific_name": species_id_name_dict[specie_id],
        })

val_df = pd.DataFrame(val_records)
assert val_df["image_path"].is_unique
print(val_df.groupby("skup").agg(slika=("image_path", "size"), vrsta=("specie_id", "nunique")).to_string())

Slika u val_ready: 29307


Od toga i u train_ready (izbacuju se): 0


      slika  vrsta
skup              
1      1000    225
2      1000    218
3      1000    224
4      1000    218
5      1000    217
6      1000    213
7      1000    212
8      1000    212
9      1000    205
10     1000    218
11     1000    227
12     1000    215
13     1000    215
14     1000    235
15     1000    218
16     1000    223
17     1000    230
18     1000    235
19     1000    215
20     1000    218
21     1000    222
22     1000    213
23     1000    207
24     1000    214
25     1000    230
26     1000    216
27     1000    225
28     1000    219
29     1000    218


In [25]:
model, _, preprocess_val = open_clip.create_model_and_transforms(MODEL_NAME)
model = model.to(DEVICE).eval()

def image_preprocessing(path):
    with Image.open(path) as img:
        tensor = preprocess_val(img.convert("RGB"))
        return tensor

@torch.no_grad()
def embeddings_generator(batch):
    features = model.encode_image(batch.to(DEVICE))
    return F.normalize(features, dim=-1).cpu()

def load_preprocess_image(record):
    try:
        return image_preprocessing(record["path"])
    except Exception as e:
        tqdm.write(f"Preskačem {record['image_path']}: {e}")
        return None

In [ ]:
VEKTORI_NPZ = BACKEND_DIR / "eval_val_vektori.npz"

cache: dict[str, np.ndarray] = {}
if VEKTORI_NPZ.exists():
    with np.load(VEKTORI_NPZ) as saved:  
        cache = dict(zip(saved["image_path"], saved["vector"]))

def save_cache():
    np.savez(VEKTORI_NPZ, image_path=np.array(list(cache)), vector=np.stack(list(cache.values())))

todo = [record for record in val_records if record["image_path"] not in cache]
print("Već sačuvano:", len(cache), "| treba izračunati:", len(todo))

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
    for i, start in enumerate(tqdm(range(0, len(todo), BATCH_SIZE), desc="Vektori")):
        batch = todo[start : start + BATCH_SIZE]
        tensors = list(pool.map(load_preprocess_image, batch))

        ok = [(record, tensor) for record, tensor in zip(batch, tensors) if tensor is not None]
        if not ok:
            continue

        vectors = embeddings_generator(torch.stack([tensor for _, tensor in ok])).numpy()
        for (record, _), vector in zip(ok, vectors):
            cache[record["image_path"]] = vector

        if i % 50 == 49:
            save_cache()

if todo and cache:
    save_cache()

query_records = [record for record in val_records if record["image_path"] in cache]
query_vectors = np.stack([cache[record["image_path"]] for record in query_records])
print("Vektora:", query_vectors.shape)

Već sačuvano: 29000 | treba izračunati: 0


Vektori: 0it [00:00, ?it/s]


Vektori: 0it [00:00, ?it/s]


Vektora: (29000, 512)


In [5]:
slika_po_vrsti = sorted((len(list(folder.glob("*.jpg"))) for folder in TRAIN_DIR.iterdir() if folder.is_dir()),
                        reverse=True)

K_GARANCIJA = slika_po_vrsti[0] + slika_po_vrsti[1] + 1

print("Najveće vrste u bazi (broj slika):", slika_po_vrsti[:5])
print("K_GARANCIJA =", K_GARANCIJA)

Najveće vrste u bazi (broj slika): [1000, 1000, 1000, 1000, 1000]
K_GARANCIJA = 2001


In [28]:
QDRANT_BATCH = 16

k_rows = []

for start in tqdm(range(0, len(query_vectors), QDRANT_BATCH), desc="Pretraga"):
    requests = [QueryRequest(query=vector.tolist(), limit=K_MAX, with_payload=["specie_id"])
                for vector in query_vectors[start : start + QDRANT_BATCH]]
    responses = client.query_batch_points(collection_name=COLLECTION, requests=requests)

    for record, response in zip(query_records[start : start + QDRANT_BATCH], responses):
        vidjene_vrste = set()
        mesto_trece = np.nan
        for mesto, point in enumerate(response.points, start=1):
            vidjene_vrste.add(point.payload["specie_id"])
            if len(vidjene_vrste) == 3:
                mesto_trece = mesto
                break
        k_rows.append({"skup": record["skup"], "potrebno_K": mesto_trece})

k_df = pd.DataFrame(k_rows)
print("Slika bez 3 vrste među K_MAX suseda:", int(k_df["potrebno_K"].isna().sum()))

Pretraga: 100%|██████████| 1813/1813 [19:24<00:00,  1.56it/s]


Slika bez 3 vrste među K_MAX suseda: 0


In [30]:
def opis(s: pd.Series) -> pd.Series:
    return pd.Series({
        "slika": len(s),
        "medijana": s.median(),
        "p95": s.quantile(0.95),
        "p99": s.quantile(0.99),
        "max": s.max(),
        "bez_3_vrste": int(s.isna().sum()),
    })

tabela_a = k_df.groupby("skup")["potrebno_K"].apply(opis).unstack()
tabela_a.loc["svi"] = opis(k_df["potrebno_K"])
print("Tabela A — mesto na kom se pojavi 3. vrsta")
print(tabela_a.round(1).to_string(), "\n")

KANDIDATI_K = [10, 50, 100, 200, 500, 1000, 1500, 2000, K_GARANCIJA, K_MAX]
tabela_b = pd.DataFrame({k: k_df.groupby("skup")["potrebno_K"].apply(lambda s: (s <= k).mean()) for k in KANDIDATI_K})
tabela_b.loc["svi"] = [(k_df["potrebno_K"] <= k).mean() for k in KANDIDATI_K]
tabela_b.columns.name = "K"
print("Tabela B — udeo slika sa bar 3 vrste")
print(tabela_b.round(3).to_string())

Tabela A — mesto na kom se pojavi 3. vrsta
        slika  medijana    p95    p99    max  bez_3_vrste
skup                                                     
1      1000.0      37.0  527.3  782.2  884.0          0.0
2      1000.0      39.5  566.1  810.0  901.0          0.0
3      1000.0      38.0  579.1  814.0  889.0          0.0
4      1000.0      37.0  540.1  754.1  935.0          0.0
5      1000.0      36.0  567.0  780.1  922.0          0.0
6      1000.0      46.0  552.0  793.1  907.0          0.0
7      1000.0      35.0  550.0  747.0  915.0          0.0
8      1000.0      36.0  555.4  776.0  877.0          0.0
9      1000.0      39.0  498.0  740.0  900.0          0.0
10     1000.0      33.0  513.1  771.1  882.0          0.0
11     1000.0      37.5  515.0  736.2  877.0          0.0
12     1000.0      37.0  564.1  792.1  901.0          0.0
13     1000.0      37.0  538.1  751.2  896.0          0.0
14     1000.0      35.5  550.4  762.0  838.0          0.0
15     1000.0      35.0  532.

In [6]:
K_EVAL = 2001
K = 200

assert K_EVAL == K_GARANCIJA

In [31]:
UPITI_CSV = BACKEND_DIR / "eval_upiti.csv"

rows = []

for start in tqdm(range(0, len(query_vectors), QDRANT_BATCH), desc="Pretraga"):
    requests = [QueryRequest(query=vector.tolist(), limit=K_EVAL, with_payload=["specie_id"])
                for vector in query_vectors[start : start + QDRANT_BATCH]]
    responses = client.query_batch_points(collection_name=COLLECTION, requests=requests)

    for record, response in zip(query_records[start : start + QDRANT_BATCH], responses):
        species_scores: dict[str, float] = {}
        for point in response.points:
            species_scores.setdefault(point.payload["specie_id"], point.score)
        ranked = list(species_scores.items())

        ranked_names = [species_id_name_dict[specie_id] for specie_id, _ in ranked]
        if record["scientific_name"] in ranked_names:
            true_rank = ranked_names.index(record["scientific_name"]) + 1
            true_score = ranked[true_rank - 1][1]
        else:
            true_rank, true_score = np.nan, np.nan

        row = {
            "skup": record["skup"],
            "image_path": record["image_path"],
            "specie_id": record["specie_id"],
            "scientific_name": record["scientific_name"],
        }
        for i, (specie_id, score) in enumerate(ranked[:3], start=1):
            row[f"top{i}_specie_id"] = specie_id
            row[f"top{i}_name"] = species_id_name_dict[specie_id]
            row[f"top{i}_score"] = score
        row["true_rank"] = true_rank
        row["true_score"] = true_score
        rows.append(row)

upiti_df = pd.DataFrame(rows)
upiti_df.to_csv(UPITI_CSV, index=False, encoding="utf-8-sig")

print("Upita:", len(upiti_df))
print("Tačna vrsta nije ni među K suseda:", int(upiti_df["true_rank"].isna().sum()))
upiti_df.drop(columns=["image_path", "top1_specie_id", "top2_specie_id", "top3_specie_id"]).round(4).head(10)

Pretraga: 100%|██████████| 1813/1813 [17:32<00:00,  1.72it/s]


Upita: 29000
Tačna vrsta nije ni među K suseda: 22


,skup,specie_id,scientific_name,top1_name,top1_score,top2_name,top2_score,top3_name,top3_score,true_rank,true_score
0,1,0002,Pelargonium graveolens,Pelargonium graveolens,0.8030,Anemone coronaria,0.6374,Althaea officinalis,0.6356,1.0,0.8030
1,1,0002,Pelargonium graveolens,Pelargonium graveolens,0.7576,Cucurbita pepo,0.7066,Papaver somniferum,0.6715,1.0,0.7576
2,1,0002,Pelargonium graveolens,Pelargonium graveolens,0.7554,Pelargonium inquinans,0.7133,Pelargonium × hybridum,0.7096,1.0,0.7554
3,1,0002,Pelargonium graveolens,Pelargonium graveolens,0.8294,Pelargonium peltatum,0.7981,Pelargonium × hybridum,0.7841,1.0,0.8294
4,1,0002,Pelargonium graveolens,Pelargonium graveolens,0.7964,Pelargonium inquinans,0.6824,Pelargonium × hybridum,0.6663,1.0,0.7964
5,1,0003,Cirsium arvense,Cirsium arvense,0.8304,Cirsium monspessulanum,0.8148,Cirsium vulgare,0.7471,1.0,0.8304
6,1,0003,Cirsium arvense,Cirsium rivulare,0.7638,Cirsium palustre,0.7568,Cirsium arvense,0.7549,3.0,0.7549
7,1,0003,Cirsium arvense,Cirsium arvense,0.8001,Cirsium vulgare,0.7468,Cirsium spinosissimum,0.7155,1.0,0.8001
8,1,0003,Cirsium arvense,Cirsium arvense,0.7982,Cirsium palustre,0.7649,Cirsium heterophyllum,0.7605,1.0,0.7982
9,1,0003,Cirsium arvense,Cirsium arvense,0.7080,Cirsium vulgare,0.6385,Lophiolepis eriophora,0.6353,1.0,0.7080


In [32]:
ranks = upiti_df["true_rank"]
N = len(upiti_df)

hr_rows = []
prev_hits = 0
for k in range(1, 11):
    in_top_k = ranks <= k
    hits = int(in_top_k.sum())
    po_skupu = in_top_k.groupby(upiti_df["skup"]).mean() * 100
    hr_rows.append({
        "k": k,
        "novih_pogodaka": hits - prev_hits,
        "hit_rate_%": hits / N * 100,
        "dobitak_pp": (hits - prev_hits) / N * 100,
        "min_po_skupu_%": po_skupu.min(),
        "max_po_skupu_%": po_skupu.max(),
    })
    prev_hits = hits

hr_df = pd.DataFrame(hr_rows).set_index("k")
print(f"Slika: {N} | tačna vrsta nije ni u top10: {N - prev_hits} ({(N - prev_hits) / N:.1%})")
hr_df.round(2)

Slika: 29000 | tačna vrsta nije ni u top10: 243 (0.8%)


,novih_pogodaka,hit_rate_%,dobitak_pp,min_po_skupu_%,max_po_skupu_%
k,,,,,
1,23233,80.11,80.11,77.1,82.6
2,3617,92.59,12.47,90.6,94.8
3,1022,96.11,3.52,94.8,97.2
4,397,97.48,1.37,96.3,98.4
5,171,98.07,0.59,97.3,98.8
6,125,98.50,0.43,97.6,99.4
7,73,98.75,0.25,97.9,99.5
8,45,98.91,0.16,98.1,99.6
9,49,99.08,0.17,98.4,99.6


In [7]:
UPITI_CSV = BACKEND_DIR / "eval_upiti.csv"
ID_KOLONE = ["specie_id", "top1_specie_id", "top2_specie_id", "top3_specie_id"]

upiti_df = pd.read_csv(UPITI_CSV, dtype={kolona: str for kolona in ID_KOLONE})

upiti_df["gap"] = upiti_df["top1_score"] - upiti_df["top2_score"]
upiti_df["top1_tacan"] = upiti_df["scientific_name"] == upiti_df["top1_name"]

L_RADNA = 3
upiti_df["tacna_u_listi"] = upiti_df["true_rank"] <= L_RADNA

print("Upita:", len(upiti_df))
print(f"Top1 tačan (bez ikakvog praga): {upiti_df['top1_tacan'].mean():.1%}")
print(f"Tačna vrsta nije ni među top{L_RADNA}: {(~upiti_df['tacna_u_listi']).sum()} "
      f"({(~upiti_df['tacna_u_listi']).mean():.1%})")
upiti_df[["scientific_name", "top1_name", "top1_score", "top2_score", "gap", "top1_tacan", "true_rank"]].round(3).head(10)

Upita: 29000
Top1 tačan (bez ikakvog praga): 80.1%
Tačna vrsta nije ni među top3: 1128 (3.9%)


,scientific_name,top1_name,top1_score,top2_score,gap,top1_tacan,true_rank
0,Pelargonium graveolens,Pelargonium graveolens,0.803,0.637,0.166,True,1.0
1,Pelargonium graveolens,Pelargonium graveolens,0.758,0.707,0.051,True,1.0
2,Pelargonium graveolens,Pelargonium graveolens,0.755,0.713,0.042,True,1.0
3,Pelargonium graveolens,Pelargonium graveolens,0.829,0.798,0.031,True,1.0
4,Pelargonium graveolens,Pelargonium graveolens,0.796,0.682,0.114,True,1.0
5,Cirsium arvense,Cirsium arvense,0.830,0.815,0.016,True,1.0
6,Cirsium arvense,Cirsium rivulare,0.764,0.757,0.007,False,3.0
7,Cirsium arvense,Cirsium arvense,0.800,0.747,0.053,True,1.0
8,Cirsium arvense,Cirsium arvense,0.798,0.765,0.033,True,1.0
9,Cirsium arvense,Cirsium arvense,0.708,0.639,0.069,True,1.0


In [8]:
OPSEZI_SKORA = [0, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 1.01]

opseg = pd.cut(upiti_df["top1_score"], OPSEZI_SKORA, right=False)
tabela_skora = upiti_df.groupby(opseg, observed=True)["tacna_u_listi"].agg(upita="size", tacna_u_listi="sum")
tabela_skora["nema_tacne_%"] = (1 - tabela_skora["tacna_u_listi"] / tabela_skora["upita"]) * 100
tabela_skora.index.name = "top1_score"
tabela_skora.round(1)

,upita,tacna_u_listi,nema_tacne_%
top1_score,,,
"[0.0, 0.5)",84,42,50.0
"[0.5, 0.55)",178,112,37.1
"[0.55, 0.6)",492,358,27.2
"[0.6, 0.65)",1081,928,14.2
"[0.65, 0.7)",2209,2020,8.6
"[0.7, 0.75)",3779,3595,4.9
"[0.75, 0.8)",5657,5502,2.7
"[0.8, 0.85)",7076,6953,1.7
"[0.85, 0.9)",6223,6163,1.0


In [9]:
KANDIDATI_PRAGA = np.round(np.arange(0.50, 0.801, 0.01), 2)

def oceni_prag(df: pd.DataFrame, prag: float) -> dict:
    odbijen = df["top1_score"] < prag
    promasaj = ~df["tacna_u_listi"]
    return {
        "PRAG": prag,
        "odbijenih": int(odbijen.sum()),
        "odbijenih_%": odbijen.mean() * 100,
        "odbijeno_s_pravom": int((odbijen & promasaj).sum()),
        "odbijeno_bez_potrebe": int((odbijen & ~promasaj).sum()),
        "s_pravom_medju_odbijenima_%": promasaj[odbijen].mean() * 100,
        "uhvaceno_promasaja_%": (odbijen & promasaj).sum() / promasaj.sum() * 100,
    }

prag_df = pd.DataFrame([oceni_prag(upiti_df, prag) for prag in KANDIDATI_PRAGA]).set_index("PRAG")
pd.set_option("display.max_rows", 40)
prag_df.round(1)

,odbijenih,odbijenih_%,odbijeno_s_pravom,odbijeno_bez_potrebe,s_pravom_medju_odbijenima_%,uhvaceno_promasaja_%
PRAG,,,,,,
0.50,84,0.3,42,42,50.0,3.7
0.51,110,0.4,54,56,49.1,4.8
0.52,139,0.5,68,71,48.9,6.0
0.53,176,0.6,79,97,44.9,7.0
0.54,214,0.7,90,124,42.1,8.0
0.55,262,0.9,108,154,41.2,9.6
0.56,334,1.2,127,207,38.0,11.3
0.57,401,1.4,151,250,37.7,13.4
0.58,510,1.8,180,330,35.3,16.0


In [10]:
MIN_S_PRAVOM = 100 / 3

def izaberi_prag(df: pd.DataFrame) -> float:
    izabran = np.nan
    for prag in KANDIDATI_PRAGA:
        if oceni_prag(df, prag)["s_pravom_medju_odbijenima_%"] >= MIN_S_PRAVOM:
            izabran = prag
    return izabran

PRAG_ABSTAIN = izaberi_prag(upiti_df)
print("PRAG_ABSTAIN =", PRAG_ABSTAIN)
print(prag_df.loc[[PRAG_ABSTAIN]].round(1).to_string(), "\n")

prag_po_skupu = pd.Series({skup: izaberi_prag(grupa) for skup, grupa in upiti_df.groupby("skup")})
print("PRAG_ABSTAIN po skupovima (vrednost: u koliko skupova je izabrana; NaN = nijedan prag ne ispunjava pravilo):")
print(prag_po_skupu.value_counts(dropna=False).sort_index().to_string())

PRAG_ABSTAIN = 0.59
      odbijenih  odbijenih_%  odbijeno_s_pravom  odbijeno_bez_potrebe  s_pravom_medju_odbijenima_%  uhvaceno_promasaja_%
PRAG                                                                                                                    
0.59        626          2.2                215                   411                         34.3                  19.1 



PRAG_ABSTAIN po skupovima (vrednost: u koliko skupova je izabrana; NaN = nijedan prag ne ispunjava pravilo):
0.50    1
0.52    1
0.53    1
0.55    3
0.57    1
0.58    4
0.59    1
0.60    4
0.61    4
0.62    2
0.63    3
0.64    1
NaN     3


In [11]:
preostali = upiti_df[upiti_df["top1_score"] >= PRAG_ABSTAIN]
print(f"Upita koji su prošli abstain: {len(preostali)} ({len(preostali) / len(upiti_df):.1%})")

OPSEZI_GAP = [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10, 1.0]

opseg_gap = pd.cut(preostali["gap"], OPSEZI_GAP, right=False)
tabela_gapa = preostali.groupby(opseg_gap, observed=True)["top1_tacan"].agg(upita="size", top1_tacnih="sum")
tabela_gapa["top1_tacan_%"] = tabela_gapa["top1_tacnih"] / tabela_gapa["upita"] * 100
tabela_gapa.index.name = "gap"
tabela_gapa.round(1)

Upita koji su prošli abstain: 28374 (97.8%)


,upita,top1_tacnih,top1_tacan_%
gap,,,
"[0.0, 0.01)",4256,2086,49.0
"[0.01, 0.02)",3621,2332,64.4
"[0.02, 0.03)",3049,2247,73.7
"[0.03, 0.04)",2529,2080,82.2
"[0.04, 0.05)",2118,1848,87.3
"[0.05, 0.06)",1878,1721,91.6
"[0.06, 0.08)",3005,2847,94.7
"[0.08, 0.1)",2185,2132,97.6
"[0.1, 1.0)",5733,5682,99.1


In [12]:
KANDIDATI_TAU = np.round(np.arange(0.0, 0.1001, 0.005), 3)

def oceni_tau(df: pd.DataFrame, tau: float) -> dict:
    siguran = df["gap"] >= tau
    tacan = df["top1_tacan"]
    return {
        "TAU": tau,
        "sigurnih_%": siguran.mean() * 100,
        "tacnost_sigurnih_%": tacan[siguran].mean() * 100,
        "sigurni_a_pogresni": int((siguran & ~tacan).sum()),
        "nesigurni_a_tacni": int((~siguran & tacan).sum()),
    }

tau_df = pd.DataFrame([oceni_tau(preostali, tau) for tau in KANDIDATI_TAU]).set_index("TAU")
tau_df.round(1)

,sigurnih_%,tacnost_sigurnih_%,sigurni_a_pogresni,nesigurni_a_tacni
TAU,,,,
0.000,100.0,81.0,5399,0
0.005,92.2,84.1,4169,996
0.010,85.0,86.6,3229,2086
0.015,78.3,88.8,2482,3242
0.020,72.2,90.5,1940,4418
0.025,66.8,92.0,1514,5544
0.030,61.5,93.5,1138,6665
0.035,56.7,94.5,886,7774
0.040,52.6,95.4,689,8745


In [13]:
CILJ_TACNOSTI = 95.0

def izaberi_tau(df: pd.DataFrame) -> float:
    for tau in KANDIDATI_TAU:
        if oceni_tau(df, tau)["tacnost_sigurnih_%"] >= CILJ_TACNOSTI:
            return tau
    return np.nan

print("Pravilo 95% daje TAU =", izaberi_tau(preostali))
tau_po_skupu = pd.Series({skup: izaberi_tau(grupa) for skup, grupa in preostali.groupby("skup")})
print("Po skupovima (vrednost: u koliko skupova je izabrana):")
print(tau_po_skupu.value_counts().sort_index().to_string(), "\n")

TAU = 0.06
print("Izabrano TAU =", TAU)
print(tau_df.loc[[TAU]].round(1).to_string())

Pravilo 95% daje TAU = 0.04


Po skupovima (vrednost: u koliko skupova je izabrana):
0.030     1
0.035     8
0.040    14
0.045     3
0.050     2
0.070     1 

Izabrano TAU = 0.06
      sigurnih_%  tacnost_sigurnih_%  sigurni_a_pogresni  nesigurni_a_tacni
TAU                                                                        
0.06        38.5                97.6                 262              12314


In [14]:
MIN_DOBITAK_PP = 2.0

nesigurni = preostali[preostali["gap"] < TAU]

hr_rows = []
prev_hits = 0
for k in range(1, 7):
    hits = int((nesigurni["true_rank"] <= k).sum())
    hr_rows.append({"k": k, "hit_rate_%": hits / len(nesigurni) * 100, "dobitak_pp": (hits - prev_hits) / len(nesigurni) * 100})
    prev_hits = hits
hr_nesigurni = pd.DataFrame(hr_rows).set_index("k")

DUZINA_LISTE = 1
for k in range(2, 7):
    if hr_nesigurni.loc[k, "dobitak_pp"] < MIN_DOBITAK_PP:
        break
    DUZINA_LISTE = k

print(f"Nesigurnih upita: {len(nesigurni)} ({len(nesigurni) / len(upiti_df):.1%} svih)")
print("DUZINA_LISTE =", DUZINA_LISTE, "| radna pretpostavka iz koraka 7 je bila", L_RADNA)
print("Krug se zatvorio." if DUZINA_LISTE == L_RADNA else "RAZLIKA: postaviti L_RADNA = DUZINA_LISTE i pokrenuti korake 7-9 ponovo.")
print()
print(hr_nesigurni.round(2).to_string())

Nesigurnih upita: 17451 (60.2% svih)
DUZINA_LISTE = 3 | radna pretpostavka iz koraka 7 je bila 3
Krug se zatvorio.

   hit_rate_%  dobitak_pp
k                        
1       70.56       70.56
2       90.00       19.44
3       95.24        5.24
4       97.17        1.93
5       97.99        0.82
6       98.48        0.49


In [15]:
def simuliraj(df: pd.DataFrame) -> pd.Series:
    odbijen = df["top1_score"] < PRAG_ABSTAIN
    siguran = ~odbijen & (df["gap"] >= TAU)
    lista = ~odbijen & ~siguran

    uspeh_siguran = df["top1_tacan"] & siguran
    uspeh_lista = (df["true_rank"] <= DUZINA_LISTE) & lista
    odgovoreno = siguran | lista

    return pd.Series({
        "odbijeno_%": odbijen.mean() * 100,
        "odbijeno_s_pravom_%": (~(df["true_rank"] <= DUZINA_LISTE))[odbijen].mean() * 100,
        "siguran_%": siguran.mean() * 100,
        "tacnost_sigurnih_%": df["top1_tacan"][siguran].mean() * 100,
        "lista_%": lista.mean() * 100,
        "tacna_u_listi_%": (df["true_rank"] <= DUZINA_LISTE)[lista].mean() * 100,
        "uspeh_odgovorenih_%": (uspeh_siguran | uspeh_lista).sum() / odgovoreno.sum() * 100,
        "uspeh_svih_upita_%": (uspeh_siguran | uspeh_lista).sum() / len(df) * 100,
    })

sistem = simuliraj(upiti_df)
po_skupu = pd.DataFrame({skup: simuliraj(grupa) for skup, grupa in upiti_df.groupby("skup")}).T

rezime = pd.DataFrame({"svih 29 000": sistem, "min po skupu": po_skupu.min(), "max po skupu": po_skupu.max()})
print(f"PRAG_ABSTAIN = {PRAG_ABSTAIN} | TAU = {TAU} | DUZINA_LISTE = {DUZINA_LISTE}\n")
rezime.round(1)

PRAG_ABSTAIN = 0.59 | TAU = 0.06 | DUZINA_LISTE = 3



,svih 29 000,min po skupu,max po skupu
odbijeno_%,2.2,1.3,3.0
odbijeno_s_pravom_%,34.3,12.5,58.3
siguran_%,37.7,34.6,40.3
tacnost_sigurnih_%,97.6,94.6,99.2
lista_%,60.2,57.7,63.2
tacna_u_listi_%,95.2,93.2,97.0
uspeh_odgovorenih_%,96.2,94.9,97.3
uspeh_svih_upita_%,94.1,92.7,95.3


In [16]:
QDRANT_BATCH = 16
sistem_rows = []

for start in tqdm(range(0, len(query_vectors), QDRANT_BATCH), desc=f"Sistem, K = {K}"):
    requests = [QueryRequest(query=vector.tolist(), limit=K, with_payload=["specie_id"])
                for vector in query_vectors[start : start + QDRANT_BATCH]]
    responses = client.query_batch_points(collection_name=COLLECTION, requests=requests)

    for record, response in zip(query_records[start : start + QDRANT_BATCH], responses):
        species_scores: dict[str, float] = {}
        for point in response.points:
            species_scores.setdefault(point.payload["specie_id"], point.score)
        ranked = list(species_scores.items())

        top1_score = ranked[0][1]
        if len(ranked) >= 2:
            gap = top1_score - ranked[1][1]
        else:
            gap = top1_score - response.points[-1].score

        if top1_score < PRAG_ABSTAIN:
            grana, prikaz = "abstain", []
        elif gap >= TAU:
            grana, prikaz = "siguran", [species_id_name_dict[ranked[0][0]]]
        else:
            grana, prikaz = "lista", [species_id_name_dict[i] for i, _ in ranked[:DUZINA_LISTE]]

        sistem_rows.append({
            "skup": record["skup"],
            "grana": grana,
            "pogodak": record["scientific_name"] in prikaz,
            "duzina": len(prikaz),
        })

sistem_df = pd.DataFrame(sistem_rows)
sistem_df.to_csv(BACKEND_DIR / "eval_sistem_k200.csv", index=False, encoding="utf-8-sig")


def mere_sistema(df: pd.DataFrame) -> pd.Series:
    siguran, lista = df["grana"] == "siguran", df["grana"] == "lista"
    odgovoreno = df["grana"] != "abstain"
    return pd.Series({
        "odbijeno_%": (~odgovoreno).mean() * 100,
        "siguran_%": siguran.mean() * 100,
        "tacnost_sigurnih_%": df.loc[siguran, "pogodak"].mean() * 100,
        "lista_%": lista.mean() * 100,
        "tacna_u_listi_%": df.loc[lista, "pogodak"].mean() * 100,
        "duzina_liste": df.loc[lista, "duzina"].mean(),
        "uspeh_odgovorenih_%": df.loc[odgovoreno, "pogodak"].mean() * 100,
        "uspeh_svih_%": df["pogodak"].mean() * 100,
    })


po_skupu_k = pd.DataFrame({skup: mere_sistema(grupa) for skup, grupa in sistem_df.groupby("skup")}).T
rezime_k = pd.DataFrame({
    "svih 29 000": mere_sistema(sistem_df),
    "min po skupu": po_skupu_k.min(),
    "max po skupu": po_skupu_k.max(),
})
rezime_k.round(2)

Sistem, K = 200:   0%|          | 0/1813 [00:00<?, ?it/s]


Sistem, K = 200:   0%|          | 1/1813 [00:01<55:42,  1.84s/it]


Sistem, K = 200:   0%|          | 2/1813 [00:02<29:14,  1.03it/s]


Sistem, K = 200:   0%|          | 3/1813 [00:02<25:11,  1.20it/s]


Sistem, K = 200:   0%|          | 4/1813 [00:03<26:12,  1.15it/s]


Sistem, K = 200:   0%|          | 5/1813 [00:04<20:03,  1.50it/s]


... (874 outputs left out)

Sistem, K = 200: 100%|█████████▉| 1809/1813 [02:12<00:00, 15.41it/s]


Sistem, K = 200: 100%|█████████▉| 1812/1813 [02:12<00:00, 16.70it/s]


Sistem, K = 200: 100%|██████████| 1813/1813 [02:12<00:00, 13.63it/s]


,svih 29 000,min po skupu,max po skupu
odbijeno_%,2.20,1.30,3.10
siguran_%,37.71,34.60,40.30
tacnost_sigurnih_%,97.59,94.66,99.23
lista_%,60.09,57.60,63.20
tacna_u_listi_%,95.23,93.17,96.96
duzina_liste,2.91,2.88,2.94
uspeh_odgovorenih_%,96.14,94.98,97.44
uspeh_svih_%,94.02,92.70,95.30


In [42]:
NISU_BILJKE_DIR = DATASET_DIR / "_nisu_biljke"
SLIKA_NASTAVCI = {".jpg", ".jpeg", ".png", ".jfif", ".webp", ".bmp", ".avif"}

kontrola_records = [
    {"path": path, "image_path": path.name}
    for path in sorted(NISU_BILJKE_DIR.iterdir())
    if path.suffix.lower() in SLIKA_NASTAVCI
]
print("Slika u folderu:", len(kontrola_records))

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
    tensors = list(pool.map(load_preprocess_image, kontrola_records))

ok = [(record, tensor) for record, tensor in zip(kontrola_records, tensors) if tensor is not None]
kontrola_records = [record for record, _ in ok]
kontrola_vectors = embeddings_generator(torch.stack([tensor for _, tensor in ok])).numpy()

print("Uspešno učitano:", len(kontrola_records))

Slika u folderu: 80
Uspešno učitano: 80


In [43]:
kontrola_rows = []

for start in range(0, len(kontrola_vectors), QDRANT_BATCH):
    requests = [QueryRequest(query=vector.tolist(), limit=K, with_payload=["specie_id"])
                for vector in kontrola_vectors[start : start + QDRANT_BATCH]]
    responses = client.query_batch_points(collection_name=COLLECTION, requests=requests)

    for record, response in zip(kontrola_records[start : start + QDRANT_BATCH], responses):
        species_scores: dict[str, float] = {}
        for point in response.points:
            species_scores.setdefault(point.payload["specie_id"], point.score)
        ranked = list(species_scores.items())

        kontrola_rows.append({
            "image_path": record["image_path"],
            "top1_name": species_id_name_dict[ranked[0][0]],
            "top1_score": ranked[0][1],
            "gap": ranked[0][1] - ranked[1][1],
        })

kontrola_df = pd.DataFrame(kontrola_rows)
kontrola_df.to_csv(BACKEND_DIR / "eval_nisu_biljke.csv", index=False, encoding="utf-8-sig")
kontrola_df.round(3).head(15)

,image_path,top1_name,top1_score,gap
0,01-Inspirational-Quotes-to-Get-You-Through-the...,Cucurbita pepo,0.358,0.001
1,015a1757e67f766a12621af543b3a5fd.jpg,Roemeria argemone,0.418,0.012
2,05-Inspirational-Quotes-to-Get-You-Through-the...,Lactuca serriola,0.393,0.029
3,1328228.png,Liriodendron tulipifera,0.575,0.023
4,14-146045_m.jpg,Pelargonium zonale,0.404,0.019
5,1600w-UnwMGJ4UsZs.webp,Acacia dealbata,0.378,0.005
6,162-1625070_uplifting-backgrounds.jpg,Alocasia macrorrhizos,0.412,0.046
7,172482-flower-spring-plant-ecoregion-people_in...,Calendula officinalis,0.685,0.008
8,1920-x-1080-nature-desktop-7uzi3zf1qeoosb63.jpg,Nymphaea alba,0.636,0.039
9,1ebccbfb6aa1f89cc7dedb349b46e378.jpg,Lactuca sativa,0.407,0.017


In [44]:
poredjenje = pd.DataFrame({
    "biljke iz baze (val)": upiti_df["top1_score"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]),
    "nisu biljke": kontrola_df["top1_score"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]),
})
print(poredjenje.round(3).to_string(), "\n")

opseg_val = pd.cut(upiti_df["top1_score"], OPSEZI_SKORA, right=False).value_counts().sort_index()
opseg_kon = pd.cut(kontrola_df["top1_score"], OPSEZI_SKORA, right=False).value_counts().sort_index()
raspodela = pd.DataFrame({
    "val_%": opseg_val / len(upiti_df) * 100,
    "nisu_biljke_%": opseg_kon / len(kontrola_df) * 100,
})
raspodela.index.name = "top1_score"
raspodela.round(1)

       biljke iz baze (val)  nisu biljke
count             29000.000       80.000
mean                  0.795        0.442
std                   0.086        0.089
min                   0.344        0.322
5%                    0.636        0.341
25%                   0.743        0.378
50%                   0.807        0.413
75%                   0.858        0.484
95%                   0.910        0.634
max                   1.000        0.685 



,val_%,nisu_biljke_%
top1_score,,
"[0.0, 0.5)",0.3,76.2
"[0.5, 0.55)",0.6,10.0
"[0.55, 0.6)",1.7,6.2
"[0.6, 0.65)",3.7,5.0
"[0.65, 0.7)",7.6,2.5
"[0.7, 0.75)",13.0,0.0
"[0.75, 0.8)",19.5,0.0
"[0.8, 0.85)",24.4,0.0
"[0.85, 0.9)",21.5,0.0


In [45]:
svi_skorovi = np.concatenate([upiti_df["top1_score"].to_numpy(), kontrola_df["top1_score"].to_numpy()])
rangovi = pd.Series(svi_skorovi).rank().to_numpy()
n_val, n_kon = len(upiti_df), len(kontrola_df)
auc = (rangovi[:n_val].sum() - n_val * (n_val + 1) / 2) / (n_val * n_kon)
print(f"AUC = {auc:.3f}\n")

odbijanje = pd.DataFrame({
    "PRAG": KANDIDATI_PRAGA,
    "odbijeno_nisu_biljke_%": [(kontrola_df["top1_score"] < prag).mean() * 100 for prag in KANDIDATI_PRAGA],
    "odbijeno_val_%": [(upiti_df["top1_score"] < prag).mean() * 100 for prag in KANDIDATI_PRAGA],
}).set_index("PRAG")
odbijanje["razlika_pp"] = odbijanje["odbijeno_nisu_biljke_%"] - odbijanje["odbijeno_val_%"]

print(f"Kod izabranog PRAG_ABSTAIN = {PRAG_ABSTAIN}:")
print(odbijanje.loc[[PRAG_ABSTAIN]].round(1).to_string(), "\n")
odbijanje.round(1)

AUC = 0.993

Kod izabranog PRAG_ABSTAIN = 0.59:
      odbijeno_nisu_biljke_%  odbijeno_val_%  razlika_pp
PRAG                                                    
0.59                    90.0             2.2        87.8 



,odbijeno_nisu_biljke_%,odbijeno_val_%,razlika_pp
PRAG,,,
0.50,76.2,0.3,76.0
0.51,77.5,0.4,77.1
0.52,77.5,0.5,77.0
0.53,80.0,0.6,79.4
0.54,83.8,0.7,83.0
0.55,86.2,0.9,85.3
0.56,87.5,1.2,86.3
0.57,87.5,1.4,86.1
0.58,90.0,1.8,88.2


In [2]:
if "kontrola_df" not in dir():
    kontrola_df = pd.read_csv(BACKEND_DIR / "eval_nisu_biljke.csv")

kontrola_df[kontrola_df["top1_score"] >= 0.6].sort_values("top1_score", ascending=False).round(3)

,image_path,top1_name,top1_score,gap
7,172482-flower-spring-plant-ecoregion-people_in...,Calendula officinalis,0.685,0.008
26,75224780_2413149085449252_2050086090513055744_...,Liriodendron tulipifera,0.659,0.026
76,tulip-background-xnwquknlg6h04b3p.jpg,Papaver somniferum,0.647,0.009
8,1920-x-1080-nature-desktop-7uzi3zf1qeoosb63.jpg,Nymphaea alba,0.636,0.039
31,9_kejpirotjesen.jpeg,Casuarina cunninghamiana,0.634,0.031
11,205108-hd-autumn-wallpaper.jpg,Liriodendron tulipifera,0.623,0.048


In [52]:
ZENODO_DIR = DATASET_DIR / "zenodo"
COMBINED_DIR = ZENODO_DIR / "images Pl@ntNet v2" / "combined"
MIN_SLIKA = 10

svetske_vrste = pd.read_csv(ZENODO_DIR / "species_metadata.csv", dtype=str)
svetske_vrste["pid"] = svetske_vrste["species_id"].str.zfill(4)

moje_vrste = set(species_id_name_dict.values())
moji_rodovi = {naziv.split()[0] for naziv in moje_vrste}

kandidati = svetske_vrste[~svetske_vrste["species"].isin(moje_vrste) & svetske_vrste["genus"].isin(moji_rodovi)]

van_baze_records = []
for _, vrsta in kandidati.iterrows():
    slike = sorted((COMBINED_DIR / vrsta["pid"]).glob("*.jpg"))
    if len(slike) < MIN_SLIKA:
        continue
    for path in slike:
        van_baze_records.append({
            "path": path,
            "image_path": f"{vrsta['pid']}/{path.name}",
            "scientific_name": vrsta["species"],
            "genus": vrsta["genus"],
        })

van_baze_df = pd.DataFrame(van_baze_records)
assert not set(van_baze_df["scientific_name"]) & moje_vrste, "neka vrsta ipak postoji u bazi"

print(f"Vrsta van baze: {van_baze_df['scientific_name'].nunique()} | rodova: {van_baze_df['genus'].nunique()} "
      f"| slika: {len(van_baze_df)}")
van_baze_df.groupby("genus").agg(vrsta=("scientific_name", "nunique"), slika=("image_path", "size")).sort_values("slika", ascending=False).head(10)

Vrsta van baze: 82 | rodova: 44 | slika: 1144


,vrsta,slika
genus,,
Sedum,6,93
Hypericum,7,86
Trifolium,5,62
Alocasia,4,54
Lupinus,3,52
Dryopteris,4,48
Phyllanthus,3,46
Acacia,3,38
Fragaria,2,36


In [57]:
van_baze_rows = []

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
    for start in tqdm(range(0, len(van_baze_records), BATCH_SIZE), desc="Vektori + pretraga"):
        batch = van_baze_records[start : start + BATCH_SIZE]
        tensors = list(pool.map(load_preprocess_image, batch))

        ok = [(record, tensor) for record, tensor in zip(batch, tensors) if tensor is not None]
        if not ok:
            continue

        embeddings = embeddings_generator(torch.stack([tensor for _, tensor in ok]))
        requests = [QueryRequest(query=embedding.tolist(), limit=K, with_payload=["specie_id"])
                    for embedding in embeddings]
        responses = client.query_batch_points(collection_name=COLLECTION, requests=requests)

        for (record, _), response in zip(ok, responses):
            species_scores: dict[str, float] = {}
            for point in response.points:
                species_scores.setdefault(point.payload["specie_id"], point.score)
            ranked = list(species_scores.items())

            top1_name = species_id_name_dict[ranked[0][0]]
            van_baze_rows.append({
                "image_path": record["image_path"],
                "prava_vrsta": record["scientific_name"],
                "top1_name": top1_name,
                "top1_score": ranked[0][1],
                "gap": ranked[0][1] - ranked[1][1],
                "isti_rod": top1_name.split()[0] == record["genus"],
            })

van_baze_upiti = pd.DataFrame(van_baze_rows)
van_baze_upiti.to_csv(BACKEND_DIR / "eval_van_baze.csv", index=False, encoding="utf-8-sig")

print("Obrađeno slika:", len(van_baze_upiti))
print(f"Sistem je vratio vrstu iz ISTOG roda: {van_baze_upiti['isti_rod'].mean():.1%}")
van_baze_upiti.round(3).head(10)

Vektori + pretraga: 100%|██████████| 18/18 [02:16<00:00,  7.60s/it]


Obrađeno slika: 1144
Sistem je vratio vrstu iz ISTOG roda: 70.5%


,image_path,prava_vrsta,top1_name,top1_score,gap,isti_rod
0,0022/41088fc398bfe70a6c052f0a89975a4c1810e6f8.jpg,Nymphaea nouchali,Nymphaea alba,0.814,0.084,True
1,0022/5afc4568aac172d9cbef65a26398d1be33e70984.jpg,Nymphaea nouchali,Nymphaea alba,0.906,0.213,True
2,0022/5b87c2e482414ff970c70fd8368dfe292fa97c05.jpg,Nymphaea nouchali,Nymphaea alba,0.849,0.109,True
3,0022/6f4559b7c3e037d1ca68fefd82e9d2cbe9e2bbfd.jpg,Nymphaea nouchali,Nymphaea alba,0.884,0.162,True
4,0022/a0de3866eda463c0e99730edb64679e66cd012ea.jpg,Nymphaea nouchali,Nymphaea alba,0.728,0.026,True
5,0022/a2d0ae4e36e2882593403a587c38a67dc9cff464.jpg,Nymphaea nouchali,Nymphaea alba,0.714,0.111,True
6,0022/a4d26fb8724c7f4ffd83f960bea98aa303bedea8.jpg,Nymphaea nouchali,Nymphaea alba,0.818,0.131,True
7,0022/a798b7c0f51318e37f34f391dd78f9bf06c23de7.jpg,Nymphaea nouchali,Nymphaea alba,0.913,0.140,True
8,0022/acff11a3729ae9705b8a722853ca475820b5fba4.jpg,Nymphaea nouchali,Nymphaea alba,0.836,0.112,True
9,0022/b0303e32454025e7dfacfd2dd4b4b120a9536965.jpg,Nymphaea nouchali,Nymphaea alba,0.845,0.133,True


In [58]:
grupe = {
    "biljke iz baze (val)": upiti_df["top1_score"],
    "biljke VAN baze": van_baze_upiti["top1_score"],
    "nisu biljke": kontrola_df["top1_score"],
}

opis = pd.DataFrame({naziv: skorovi.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]) for naziv, skorovi in grupe.items()})
print(opis.round(3).to_string(), "\n")

raspodela = pd.DataFrame({
    naziv: pd.cut(skorovi, OPSEZI_SKORA, right=False).value_counts().sort_index() / len(skorovi) * 100
    for naziv, skorovi in grupe.items()
})
raspodela.index.name = "top1_score"
raspodela.round(1)

       biljke iz baze (val)  biljke VAN baze  nisu biljke
count             29000.000         1144.000       80.000
mean                  0.795            0.731        0.442
std                   0.086            0.090        0.089
min                   0.344            0.411        0.322
5%                    0.636            0.584        0.341
25%                   0.743            0.669        0.378
50%                   0.807            0.737        0.413
75%                   0.858            0.798        0.484
95%                   0.910            0.867        0.634
max                   1.000            1.000        0.685 



,biljke iz baze (val),biljke VAN baze,nisu biljke
top1_score,,,
"[0.0, 0.5)",0.3,1.0,76.2
"[0.5, 0.55)",0.6,1.8,10.0
"[0.55, 0.6)",1.7,4.4,6.2
"[0.6, 0.65)",3.7,12.3,5.0
"[0.65, 0.7)",7.6,15.8,2.5
"[0.7, 0.75)",13.0,20.5,0.0
"[0.75, 0.8)",19.5,19.9,0.0
"[0.8, 0.85)",24.4,16.7,0.0
"[0.85, 0.9)",21.5,5.9,0.0


In [59]:
def auc_protiv_vala(skorovi: pd.Series) -> float:
    spojeno = np.concatenate([upiti_df["top1_score"].to_numpy(), skorovi.to_numpy()])
    rangovi = pd.Series(spojeno).rank().to_numpy()
    n_val, n_grupa = len(upiti_df), len(skorovi)
    return (rangovi[:n_val].sum() - n_val * (n_val + 1) / 2) / (n_val * n_grupa)

print(f"AUC (biljke van baze)  = {auc_protiv_vala(van_baze_upiti['top1_score']):.3f}")
print(f"AUC (nisu biljke)      = {auc_protiv_vala(kontrola_df['top1_score']):.3f}\n")

odbijanje = pd.DataFrame({
    "PRAG": KANDIDATI_PRAGA,
    "odbijeno_van_baze_%": [(van_baze_upiti["top1_score"] < prag).mean() * 100 for prag in KANDIDATI_PRAGA],
    "odbijeno_nisu_biljke_%": [(kontrola_df["top1_score"] < prag).mean() * 100 for prag in KANDIDATI_PRAGA],
    "odbijeno_val_%": [(upiti_df["top1_score"] < prag).mean() * 100 for prag in KANDIDATI_PRAGA],
}).set_index("PRAG")

print(f"Kod izabranog PRAG_ABSTAIN = {PRAG_ABSTAIN}:")
print(odbijanje.loc[[PRAG_ABSTAIN]].round(1).to_string(), "\n")
odbijanje.round(1)

AUC (biljke van baze)  = 0.704
AUC (nisu biljke)      = 0.993

Kod izabranog PRAG_ABSTAIN = 0.59:
      odbijeno_van_baze_%  odbijeno_nisu_biljke_%  odbijeno_val_%
PRAG                                                             
0.59                  5.9                    90.0             2.2 



,odbijeno_van_baze_%,odbijeno_nisu_biljke_%,odbijeno_val_%
PRAG,,,
0.50,1.0,76.2,0.3
0.51,1.2,77.5,0.4
0.52,1.3,77.5,0.5
0.53,1.5,80.0,0.6
0.54,2.2,83.8,0.7
0.55,2.8,86.2,0.9
0.56,3.3,87.5,1.2
0.57,3.9,87.5,1.4
0.58,4.6,90.0,1.8


In [18]:
if "van_baze_upiti" not in dir():
    van_baze_upiti = pd.read_csv(BACKEND_DIR / "eval_van_baze.csv")

prosli = van_baze_upiti["top1_score"] >= PRAG_ABSTAIN
siguran = prosli & (van_baze_upiti["gap"] >= TAU)

print(f"Slika vrsta van baze: {len(van_baze_upiti)}")
print(f"  odbijeno (abstain):      {(~prosli).sum():5d}  ({(~prosli).mean():.1%})")
print(f"  prošlo -> lista:         {(prosli & ~siguran).sum():5d}  ({(prosli & ~siguran).mean():.1%})")
print(f"  prošlo -> SIGURAN top1:  {siguran.sum():5d}  ({siguran.mean():.1%})   <- najgori ishod\n")

prosli_val = upiti_df["top1_score"] >= PRAG_ABSTAIN
siguran_val = prosli_val & (upiti_df["gap"] >= TAU)
print("Za poređenje, val slike (vrsta JESTE u bazi):")
print(f"  odbijeno: {(~prosli_val).mean():.1%} | lista: {(prosli_val & ~siguran_val).mean():.1%} | siguran: {siguran_val.mean():.1%}")

Slika vrsta van baze: 1144
  odbijeno (abstain):         67  (5.9%)
  prošlo -> lista:           735  (64.2%)
  prošlo -> SIGURAN top1:    342  (29.9%)   <- najgori ishod

Za poređenje, val slike (vrsta JESTE u bazi):
  odbijeno: 2.2% | lista: 60.2% | siguran: 37.7%


In [17]:
QDRANT_BATCH = 16
K_KANDIDATI = [20, 50, 100, 200, 500, 1000, 2001]
N_UPITA = 3000

rng_k = np.random.default_rng(7)
uzorak = rng_k.choice(len(query_vectors), N_UPITA, replace=False)
uzorak_vektori = query_vectors[uzorak]
prave_vrste = [query_records[i]["scientific_name"] for i in uzorak]


def izmeri_k(k: int) -> dict:
    grane, prikazi, trajanje = [], [], 0.0

    for start in range(0, len(uzorak_vektori), QDRANT_BATCH):
        requests = [QueryRequest(query=vector.tolist(), limit=k, with_payload=["specie_id"])
                    for vector in uzorak_vektori[start : start + QDRANT_BATCH]]
        t0 = time.perf_counter()
        responses = client.query_batch_points(collection_name=COLLECTION, requests=requests)
        trajanje += time.perf_counter() - t0

        for response in responses:
            species_scores: dict[str, float] = {}
            for point in response.points:
                species_scores.setdefault(point.payload["specie_id"], point.score)
            ranked = list(species_scores.items())

            top1_score = ranked[0][1]
            if len(ranked) >= 2:
                gap = top1_score - ranked[1][1]
            else:
                gap = top1_score - response.points[-1].score

            if top1_score < PRAG_ABSTAIN:
                grane.append("abstain"); prikazi.append([])
            elif gap >= TAU:
                grane.append("siguran"); prikazi.append([species_id_name_dict[ranked[0][0]]])
            else:
                grane.append("lista"); prikazi.append([species_id_name_dict[i] for i, _ in ranked[:DUZINA_LISTE]])

    grane = np.array(grane)
    pogodak = np.array([prava in prikaz for prava, prikaz in zip(prave_vrste, prikazi)])
    duzine = np.array([len(prikaz) for prikaz in prikazi])
    siguran, lista, odgovoreno = grane == "siguran", grane == "lista", grane != "abstain"

    return {
        "K": k,
        "ms_po_upitu": trajanje / len(uzorak_vektori) * 1000,
        "siguran_%": siguran.mean() * 100,
        "tacnost_sigurnih_%": pogodak[siguran].mean() * 100,
        "lista_%": lista.mean() * 100,
        "tacna_u_listi_%": pogodak[lista].mean() * 100,
        "duzina_liste": duzine[lista].mean(),
        "uspeh_odgovorenih_%": pogodak[odgovoreno].mean() * 100,
    }


k_izbor_df = pd.DataFrame([izmeri_k(k) for k in K_KANDIDATI]).set_index("K")
k_izbor_df.round(2)

,ms_po_upitu,siguran_%,tacnost_sigurnih_%,lista_%,tacna_u_listi_%,duzina_liste,uspeh_odgovorenih_%
K,,,,,,,
20,2.84,20.57,94.49,77.23,96.16,2.11,95.81
50,3.51,31.93,96.14,65.87,95.80,2.57,95.91
100,3.48,36.63,96.63,61.17,95.53,2.81,95.94
200,4.37,38.03,96.76,59.77,95.48,2.92,95.98
500,7.09,38.03,96.76,59.77,95.48,2.98,95.98
1000,11.51,37.97,96.93,59.83,95.43,3.00,96.01
2001,19.54,38.00,96.84,59.80,95.48,3.00,96.01
